# 02. Function Calling과 도구

01에서 Agent가 모델 호출과 도구 실행을 번갈아 한다는 것을 보았습니다. 이번에는 그 "도구"를 가장 작은 것부터 직접 만듭니다. 한 줄짜리 함수에서 시작해, 인자가 늘고, 값에 제한이 붙고, 도구가 세 개가 되고, 도구가 오류를 돌려주는 순서로 한 칸씩 올라갑니다. 핵심은 한 문장입니다. **모델은 요청하고, 우리 코드가 실행한다.**

셀 첫 줄의 표지는 다음 뜻입니다.

- `# [따라하기]` 그대로 실행하고 출력을 읽습니다.
- `# [바꿔보기]` 무엇을 바꿀지 적혀 있습니다. 실행 전에 결과를 먼저 예측해 보세요.
- `# [깨뜨리기]` 일부러 실패시킵니다. 오류 메시지를 읽는 연습입니다.
- `# [내 것 만들기]` 예시가 채워져 있어 그대로 돌아갑니다. 값을 내 것으로 바꿔 보세요.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 모델은 요청하고 애플리케이션이 실행한다: 네 단계 왕복 | 6 |
| 2 | 첫 도구 | 5 |
| 3 | 모델이 보는 것: 이름, 설명, 스키마 | 4 |
| 4 | 인자 두 개, 값이 제한된 도구 | 8 |
| 5 | 도구를 고르는 기준은 설명이다 | 4 |
| 6 | 세 도구 중 고르기, 설명 바꾸기 | 8 |
| 7 | 실패 세 종류와 고칠 단서가 있는 오류 | 5 |
| 8 | 오류를 돌려주는 도구 | 6 |
| 9 | 직접 해보기: 내 도구 | 10 |
| 10 | 권한 경계, `tool_choice`, 정리와 확인 질문 | 4 |

## 호출 규모

이 노트북은 `Day02_new/.env`의 키로 실제 Claude 모델을 부릅니다. 키 없이 도는 경로는 없습니다. 모델을 부르는 셀은 실행 직전에 `announce`가 호출 규모를 한 줄로 알려 줍니다. 모델 호출이 없는 셀은 표지 옆에 "무료"라고 적었습니다. 전체 모델 호출은 20회 안팎이고, Agent 셀은 도구를 몇 번 부르느냐에 따라 조금 달라집니다. 같은 셀을 다시 돌리면 모델의 선택이 달라질 수 있습니다. 이 노트북의 출력은 한 번의 실행 기록이지 정확도 측정이 아닙니다.

In [ ]:
from rag_lab.settings import announce, key_status, make_model, require_key

print(key_status())
settings = require_key()  # 키가 없으면 여기서 한국어 안내와 함께 멈춥니다.
print("모델:", settings.model, "| effort:", settings.effort)

키는 "설정됨"인지만 확인합니다. 값은 출력하지 않습니다. "미설정"이면 `Day02_new/.env.example`을 보고 `.env`를 직접 채우세요.

## 1절. 모델은 요청하고 애플리케이션이 실행한다

모델은 문자열을 만들 뿐입니다. 파일을 읽거나 문서를 찾으려면 누군가 실제로 함수를 실행해야 합니다. 도구 호출은 네 단계 왕복입니다.

```text
1. 애플리케이션 -> 모델   도구 설명(이름, 설명, 인자 스키마)과 질문을 보낸다
2. 모델 -> 애플리케이션   AIMessage.tool_calls = [{name, args, id}]  (요청이지 실행이 아니다)
3. 애플리케이션           요청을 검사하고 허용하면 함수를 실행한다
4. 애플리케이션 -> 모델   ToolMessage(결과, tool_call_id=id)를 넣어 다시 부른다
```

모델이 직접 실행하지 못하는 이유는 모델이 글자를 만드는 프로그램일 뿐 우리 컴퓨터의 함수에 닿지 못하기 때문입니다. 그래서 모델은 "이 함수를 이 인자로 불러 주세요"라는 요청서만 쓰고, 실행할지는 우리 코드가 정합니다. 여기서 스키마란 도구가 받는 인자의 이름, 타입, 허용 값을 적은 JSON 명세서입니다. 모델은 이것만 보고 인자를 채웁니다.

2단계와 3단계 사이에 사람이 쓴 코드가 있습니다. 검사, 거부, 예산 차감이 모두 이 틈에서 일어납니다. 이제 가장 작은 도구로 이 왕복을 직접 봅니다.

## 2절. 첫 도구

`@tool`은 평범한 함수를 도구로 바꿉니다. 먼저 모델 없이 도구 하나를 만들고 직접 불러 봅니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
from langchain_core.tools import tool


@tool
def add_ten(n: int) -> int:
    """숫자 n에 10을 더한다."""
    return n + 10


print(add_ten.name, "|", add_ten.description, "|", add_ten.args)
print(add_ten.invoke({"n": 7}))

함수 이름이 `name`, docstring이 `description`, 타입 힌트가 `args`가 되었습니다. 마지막 줄은 우리가 직접 부른 결과이고, 이 호출에는 모델이 없습니다. 이제 이 도구를 모델에게 보여 주면 모델이 무엇을 내는지 봅니다.

In [ ]:
# [따라하기] 유료, 모델 호출 1회
announce("2-1 도구 요청 받기", model_calls=1)
bound = make_model().bind_tools([add_ten])
ai = bound.invoke("add_ten 도구로 7에 10을 더해 주세요.")

print("tool_calls:", ai.tool_calls)
print("stop_reason:", ai.response_metadata.get("stop_reason"))
print("본문:", repr(ai.text))

`tool_calls`에 이름, 인자, `id`가 들어 있고 `stop_reason`은 모델이 도구 결과를 기다리며 멈췄음을 말합니다. 함수는 아직 한 번도 실행되지 않았습니다. 모델은 요청서만 썼습니다. 이어서 우리가 실행하고 결과를 돌려주어 왕복을 끝냅니다.

In [ ]:
# [따라하기] 유료, 모델 호출 1회
from langchain_core.messages import HumanMessage

announce("2-2 왕복 끝내기", model_calls=1)
question = HumanMessage("add_ten 도구로 7에 10을 더해 주세요.")
result = add_ten.invoke(ai.tool_calls[0])          # 3단계: 우리가 실행한다
messages = [question, ai, result]
final = bound.invoke(messages)                      # 4단계: 결과를 넣어 다시 부른다

print("요청 id      :", ai.tool_calls[0]["id"])
print("결과 짝 id   :", result.tool_call_id, "| 결과:", result.content)
print("최종 답      :", final.text)

`ToolMessage`의 `tool_call_id`가 요청의 `id`와 같습니다. `ToolMessage`는 "요청한 도구를 돌려 보니 이런 결과였다"를 모델에게 알리는 메시지입니다. 모델은 이것을 읽어야 최종 답을 쓸 수 있고, 이 메시지를 넣지 않으면 모델은 결과를 모른 채 같은 요청을 되풀이합니다. 이 짝이 있어야 요청이 여러 개일 때도 결과가 뒤섞이지 않습니다. 또 모델은 기억이 없어서, 두 번째 호출에 `messages`를 통째로 보내야 앞 일을 압니다. 메시지 목록이 곧 기억입니다.

## 3절. 모델이 보는 것: 이름, 설명, 스키마

모델이 받는 것은 함수 본문이 아니라 이름, 설명, 인자 스키마입니다. 타입 힌트를 바꾸면 모델에게 가는 JSON이 어떻게 바뀌는지 봅니다. 먼저 예측하세요. `n: int`를 `n: str`로 바꾸면 JSON의 어느 부분이 달라질까요?

In [ ]:
# [바꿔보기] 무료, 모델 호출 0회 - 바꿀 값: 타입 힌트 n: int -> n: str (본문은 int(n) + 10)
import json
from langchain_core.utils.function_calling import convert_to_openai_tool


@tool
def add_ten_str(n: str) -> int:
    """숫자 n에 10을 더한다."""
    return int(n) + 10


print("int 버전:", json.dumps(convert_to_openai_tool(add_ten)["function"]["parameters"], ensure_ascii=False))
print("str 버전:", json.dumps(convert_to_openai_tool(add_ten_str)["function"]["parameters"], ensure_ascii=False))
print("본문 포함 여부:", "int(n)" in json.dumps(convert_to_openai_tool(add_ten_str)))

예측과 비교하세요. 달라진 곳은 `parameters` 안의 `type` 하나입니다. 함수 본문은 그 JSON에 들어 있지 않습니다. 모델은 이 도구가 무엇을 받는지만 알고 어떻게 동작하는지는 모릅니다. 그래서 이름과 설명이 곧 프롬프트입니다. 좋은 설명은 세 가지를 말합니다. 무엇을 하는가, 언제 쓰는가, 언제 쓰지 않는가. 인자는 형식과 허용 범위를 적습니다. 설명이 중요한 이유는 모델이 도구를 고를 때 읽을 수 있는 것이 이 글뿐이기 때문입니다. 설명이 모호하면 모델은 비슷한 도구 사이에서 헤맵니다.

## 4절. 인자 두 개, 값이 제한된 도구

In [ ]:
# [따라하기] 유료, 모델 호출 1회
@tool
def add(a: int, b: int) -> int:
    """두 정수 a와 b를 더한다."""
    return a + b


announce("4-1 인자 두 개", model_calls=1)
ai = make_model().bind_tools([add]).invoke("add 도구로 7과 5를 더해 주세요.")
print("tool_calls:", ai.tool_calls)

`args`에 `a`와 `b` 두 값이 담겼는지 읽으세요. 질문의 숫자가 인자로 옮겨진 것은 모델의 일입니다. 이번에는 값 하나를 빼고 물어봅니다. 예측하세요. 모델은 도구를 부를까요, 되물을까요, 값을 지어낼까요?

In [ ]:
# [바꿔보기] 유료, 모델 호출 1회 - 바꿀 값: 질문을 "7에 어떤 수를 더해 주세요"로
announce("4-2 값이 빠진 질문", model_calls=1)
ai = make_model().bind_tools([add]).invoke("add 도구로 7에 어떤 수를 더해 주세요.")
print("tool_calls:", ai.tool_calls)
print("본문:", ai.text)

`tool_calls`가 비었는지, 본문이 되묻는지, 아니면 `b`에 무언가를 채워 넣었는지 확인하세요. 값을 채웠다면 모델이 지어낸 것입니다. 이번 실행의 결과를 적어 두고, 같은 셀을 한 번 더 돌려 같은지 보세요. 이제 도구를 직접 깨뜨려 오류를 읽어 봅니다. 오류 메시지는 겁낼 대상이 아니라 읽을 자료입니다. 어느 인자가, 왜 거절되었는지가 거기에 적혀 있고, 좋은 도구는 이 정보를 모델에게도 같은 방식으로 돌려줍니다. 이 감각이 7절과 8절의 바탕입니다. 읽는 순서는 예외 이름, 인자 이름, 이유 한 줄입니다.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회 - 필수 인자 b를 뺀다. 손으로 만든 입력이며 모델이 낸 것이 아니다.
try:
    add.invoke({"a": 7})
except Exception as error:
    print(type(error).__name__)
    print(error)

`ValidationError`의 어느 줄이 빠진 인자를 말하는지 찾으세요. 함수 본문이 돌기 전에 스키마가 먼저 막았습니다. 이번에는 값에 제한을 거는 도구입니다. 회의실은 A, B, C뿐이고 시간은 1에서 4까지입니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
from typing import Literal
from pydantic import BaseModel, Field


class RoomInput(BaseModel):
    room: Literal["A", "B", "C"] = Field(description="회의실 이름")
    hours: int = Field(default=1, ge=1, le=4, description="예약 시간. 1에서 4")


@tool("book_room", args_schema=RoomInput)
def book_room(room: str, hours: int = 1) -> str:
    """회의실을 예약한다."""
    return f"{room} 회의실 {hours}시간 예약 완료"


print(json.dumps(book_room.args, ensure_ascii=False, indent=1))
print(book_room.invoke({"room": "A", "hours": 2}))

`enum`, `minimum`, `maximum`이 스키마에 들어 있습니다. 이것도 모델이 읽는 설명의 일부입니다. 범위 밖 값을 직접 넣으면 어떻게 될까요?

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회 - 범위 밖 값. 손으로 만든 입력이다.
try:
    book_room.invoke({"room": "Z", "hours": 9})
except Exception as error:
    print(type(error).__name__)
    print(error)

오류가 두 값(`room`, `hours`)을 함께 지적하는지 읽으세요. 그렇다면 모델이 범위 밖 값을 내면 어떻게 될까요? 예측하세요. 값을 그대로 낼까요, 고칠까요, 되물을까요?

In [ ]:
# [바꿔보기] 유료, 모델 호출 1회 - 바꿀 값: 질문을 "Z 회의실을 9시간 잡아 주세요"로
announce("4-3 범위 밖 요청", model_calls=1)
ai = make_model().bind_tools([book_room]).invoke("Z 회의실을 9시간 잡아 주세요.")
print("tool_calls:", ai.tool_calls)
print("본문:", ai.text)

`tool_calls`에 범위 밖 값이 그대로 있는지, 고쳐 넣었는지, 비어 있고 되묻는지 확인하세요. 어느 쪽이든 원칙은 같습니다. **모델이 낸 인자도 입력이지 사실이 아닙니다.** 그래서 우리 쪽 스키마가 한 번 더 검사합니다.

## 5절. 도구를 고르는 기준은 설명이다

도구가 둘 이상이면 모델은 이름과 설명을 읽고 고릅니다. 먼저 이름과 설명이 어울리는 두 도구를 줍니다.

In [ ]:
# [따라하기] 유료, 모델 호출 1회
@tool("search_notice", description="공지사항을 찾는다.")
def search_notice(keyword: str) -> str:
    return f"공지에서 {keyword} 검색"


@tool("search_policy", description="사내 규정을 찾는다.")
def search_policy(keyword: str) -> str:
    return f"규정에서 {keyword} 검색"


announce("5-1 설명을 보고 고르기", model_calls=1)
ai = make_model().bind_tools([search_notice, search_policy]).invoke("연차 규정 알려 주세요. 도구는 하나만 부르세요.")
print([(c["name"], c["args"]) for c in ai.tool_calls])

고른 이름을 읽으세요. 이제 함수 이름은 그대로 두고 두 설명만 맞바꿉니다. 예측하세요. 모델은 이름을 따를까요, 설명을 따를까요?

In [ ]:
# [바꿔보기] 유료, 모델 호출 1회 - 바꿀 값: 두 도구의 description을 서로 맞바꾼다
@tool("search_notice", description="사내 규정을 찾는다.")
def search_notice_swapped(keyword: str) -> str:
    return f"공지에서 {keyword} 검색"


@tool("search_policy", description="공지사항을 찾는다.")
def search_policy_swapped(keyword: str) -> str:
    return f"규정에서 {keyword} 검색"


announce("5-2 설명을 맞바꾸기", model_calls=1)
ai = make_model().bind_tools([search_notice_swapped, search_policy_swapped]).invoke("연차 규정 알려 주세요. 도구는 하나만 부르세요.")
print([(c["name"], c["args"]) for c in ai.tool_calls])

앞 셀과 같은 이름을 골랐나요, 달랐나요? 모델이 둘 다 불렀다면 이번 실행은 아무것도 가르지 못한 것입니다. 달랐다면 모델이 설명을 따른 것이고, 같았다면 이번에는 이름을 따랐거나 질문의 단어를 따른 것입니다. 어느 쪽이든 1회 관찰이므로 결론으로 삼지 말고, 값이 같게 나와도 "설명이 선택을 바꾸지 않았다"가 아니라 "이번 한 번은 바꾸지 않았다"고 읽으세요. 도구 설명은 프롬프트입니다. 관찰 메모를 한 줄 남기세요: 두 셀에서 고른 이름, 그리고 같은 셀을 한 번 더 돌렸을 때 같았는지.

## 6절. 세 도구 중 고르기, 설명 바꾸기

도구를 셋으로 늘립니다. 메뉴, 버스, 회의실입니다.

In [ ]:
# [따라하기] 유료, 모델 호출 1회
@tool
def get_menu(day: str) -> str:
    """요일(예: 월, 화)의 구내식당 점심 메뉴를 돌려준다."""
    return {"월": "김치찌개", "화": "비빔밥"}.get(day, "메뉴 없음")


@tool
def get_bus_time(stop: str) -> str:
    """정류장 이름을 받아 다음 버스 도착 시각을 돌려준다."""
    return "08:15"


three = [get_menu, get_bus_time, book_room]
model3 = make_model().bind_tools(three)
announce("6-1 세 도구 중 고르기", model_calls=1)
ai = model3.invoke("월요일 점심 메뉴가 뭐예요?")
print([(c["name"], c["args"]) for c in ai.tool_calls])

어느 도구를 어떤 인자로 골랐는지 읽으세요. 이제 질문 세 개를 바꿔 넣습니다. 먼저 표에 예측을 적으세요. 고를 도구, 한 응답에 요청이 몇 개인지, 도구가 아예 없는지입니다.

In [ ]:
# [바꿔보기] 유료, 모델 호출 3회 - 바꿀 값: 질문 세 개 (버스 / 메뉴와 버스 동시 / 대한민국 수도)
announce("6-2 질문 세 개", model_calls=3)
for q in ["강남역 정류장 버스는 언제 와요?",
          "월요일 점심 메뉴와 강남역 정류장 버스 시간을 같이 알려 주세요.",
          "대한민국의 수도는 어디예요?"]:
    ai = model3.invoke(q)
    print(q)
    print("  요청 수:", len(ai.tool_calls), [c["name"] for c in ai.tool_calls], "| 본문 있음:", bool(ai.text))

예측과 비교하세요. 동시에 묻는 질문에서 요청이 한 응답에 여러 개 들어오는지(병렬 호출), 도구가 필요 없는 질문에서 `tool_calls`가 비어 모델이 직접 답하는지 보세요. 이것이 `auto`입니다. 모델이 도구를 쓸지 말지까지 정합니다. 요청이 여러 개면 각각에 `id`가 붙고 우리가 모두 실행해 `ToolMessage`를 하나씩 돌려줍니다.

## 7절. 실패 세 종류와 고칠 단서가 있는 오류

도구가 실패하는 길은 세 가지입니다. 첫째는 없는 도구를 부르는 경우입니다. 모델이 이름을 잘못 쓰거나 보이지 않는 도구를 요청하면, 우리 쪽에서 실행하지 않고 "그런 도구가 없다"는 결과를 돌려주면 됩니다. 둘째는 인자 오류로, 스키마 검증이 함수 본문 전에 막습니다. 셋째는 실행 오류로, 함수가 돌다가 예외를 던지는 경우입니다. 세 경우 모두 모델이 다음 행동을 정할 수 있도록 무엇이 틀렸고 무엇이 맞는지가 결과에 들어 있어야 합니다.

4절에서 인자 오류를 봤습니다. 실행 오류는 어떤가요? 오류가 `invoke` 밖으로 나가면 Agent 전체가 멈춥니다. 반대로 `status`와 다음 행동을 담아 돌려주면 모델이 읽고 스스로 고칠 수 있습니다.

## 8절. 오류를 돌려주는 도구

없는 요일을 물으면 빈 문자열이나 "없음"이 아니라, 무엇이 틀렸고 무엇이 맞는지 알려 줍니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
VALID_DAYS = ["월", "화", "수", "목", "금"]
MENU = {"월": "김치찌개", "화": "비빔밥", "수": "제육볶음", "목": "된장찌개", "금": "돈가스"}


@tool
def get_menu_checked(day: str) -> dict:
    """요일(월~금)의 구내식당 점심 메뉴를 돌려준다. 주말에는 식당을 운영하지 않는다."""
    if day not in MENU:
        return {"status": "unknown_day", "valid_days": VALID_DAYS,
                "next_action": "valid_days 중 하나로 다시 부르거나, 주말은 운영하지 않는다고 답한다."}
    return {"status": "ok", "menu": MENU[day]}


print(get_menu_checked.invoke({"day": "월"}))
print(get_menu_checked.invoke({"day": "일"}))

없는 요일의 결과가 빈 값과 무엇이 다른지 보세요. `status`가 실패를 말하고, `valid_days`와 `next_action`이 고칠 단서입니다. 이 도구를 모델에게 맡겨 봅니다. 모델이 `status`를 읽고 무엇을 하는지 봅니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 2회
from deepagents import create_deep_agent

announce("8-1 오류를 읽는 Agent", model_calls=2)
agent = create_deep_agent(model=make_model(), tools=[get_menu_checked],
                          system_prompt="구내식당 메뉴를 도구로 확인해 한국어로 답한다.")
state = agent.invoke({"messages": [("user", "일요일 메뉴 알려 주세요.")]})
for m in state["messages"]:
    kind = type(m).__name__
    calls = [(c["name"], c["args"]) for c in getattr(m, "tool_calls", [])]
    print(f"{kind:12} {calls or ''} {m.text[:150]!r}")

메시지를 줄별로 읽으세요. 도구가 `unknown_day`를 돌려준 뒤 모델이 `next_action`대로 답했는지, 다른 요일로 다시 불렀는지 봅니다. 이제 반대로, 오류를 돌려주지 않고 예외를 던지는 도구를 만들어 깨뜨립니다. 아래 두 셀의 요청은 **손으로 만든 것**입니다. 모델이 틀린 요청을 낼 때까지 기다릴 수 없어서 직접 지어 넣는 장면이며, 이 노트북에서 손으로 만든 메시지는 [깨뜨리기] 셀에만 있습니다.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회 - 손으로 만든 요청(모델이 낸 것이 아님). 예외가 invoke 밖으로 나온다.
@tool
def fragile_lookup(team: str) -> str:
    """팀 담당자를 찾는다."""
    return {"개발": "김민수"}[team]


request = {"name": "fragile_lookup", "args": {"team": "없는팀"}, "id": "call_fragile", "type": "tool_call"}
try:
    fragile_lookup.invoke(request)
except Exception as error:
    print("invoke 밖으로 나온 예외:", type(error).__name__, error)

`KeyError`가 그대로 밖으로 나왔습니다. Agent 루프 안이라면 실행 전체가 여기서 멈춥니다. 이제 `ToolException`과 `handle_tool_error`로 고칩니다.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회 - 손으로 만든 요청(모델이 낸 것이 아님). 예외를 ToolMessage로 바꾼다.
from langchain_core.tools import ToolException


@tool
def safe_lookup(team: str) -> str:
    """팀 담당자를 찾는다. 팀 이름은 개발 하나만 있다."""
    owners = {"개발": "김민수"}
    if team not in owners:
        raise ToolException(f"'{team}' 팀이 없습니다. 가능한 팀: {list(owners)}. 가능한 팀 중 하나로 다시 부르세요.")
    return owners[team]


safe_lookup.handle_tool_error = True  # ToolException을 밖으로 던지지 않고 ToolMessage로 바꾼다
request = {"name": "safe_lookup", "args": {"team": "없는팀"}, "id": "call_safe", "type": "tool_call"}
reply = safe_lookup.invoke(request)
print(type(reply).__name__, "| status:", reply.status)
print(reply.content)

이번에는 실행이 멈추지 않고 `status`가 `error`인 `ToolMessage`가 돌아왔습니다. 내용에 고칠 단서(가능한 팀 목록, 다시 부르라는 말)가 있는지 읽으세요. 모델은 이것을 받아 인자를 고쳐 다시 부를 수 있습니다.

## 직접 해보기: 내 도구 (9절)

이제 내 도구를 만듭니다. 아래 세 셀은 예시가 채워져 있어 위에서 아래로 그대로 돕니다. 값을 내 것으로 바꿔 다시 실행하세요. 입력을 비우면 안내가 출력되고 모델은 부르지 않습니다. 도구는 작을수록 좋습니다. 이름은 동사와 대상이 보이게, 설명은 언제 쓰고 언제 쓰지 않는지가 보이게, 오류는 status와 고칠 단서가 있게 만드는 것이 오늘 배운 기준입니다.

In [ ]:
# [내 것 만들기] 무료, 모델 호출 0회 - 아래 사전을 내 데이터로 바꾸세요
MY_TOOL_DATA = {"개발팀": "김민수", "인사팀": "이서연", "재무팀": "박지훈"}   # 팀별 담당자 예시

if not MY_TOOL_DATA:
    print("MY_TOOL_DATA가 비어 있습니다. {'키': '값'} 형태로 채우세요. 예: {'개발팀': '김민수'}")
else:
    @tool
    def lookup_mine(key: str) -> dict:
        """MY_TOOL_DATA에서 key의 값을 찾는다. 없으면 status와 가능한 키를 돌려준다."""
        if key not in MY_TOOL_DATA:
            return {"status": "unknown_key", "valid_keys": list(MY_TOOL_DATA),
                    "next_action": "valid_keys 중 하나로 다시 부른다."}
        return {"status": "ok", "value": MY_TOOL_DATA[key]}

    print(lookup_mine.invoke({"key": next(iter(MY_TOOL_DATA))}))
    print(lookup_mine.invoke({"key": "없는키"}))

있는 키와 없는 키의 `status` 구조를 비교하세요. 다음 셀은 질문 하나를 모델에게 주고 `tool_calls`만 읽습니다.

In [ ]:
# [내 것 만들기] 유료, 모델 호출 1회 - MY_QUESTION을 내 질문으로 바꾸세요
MY_QUESTION = "인사팀 담당자가 누구예요?"

if not MY_QUESTION.strip() or not MY_TOOL_DATA:
    print("MY_QUESTION과 MY_TOOL_DATA를 채우세요. 질문은 내 데이터로 답할 수 있는 한 문장이 좋습니다.")
else:
    announce("9-2 내 질문", model_calls=1)
    ai = make_model().bind_tools([lookup_mine]).invoke(MY_QUESTION)
    print("tool_calls:", [(c["name"], c["args"]) for c in ai.tool_calls])
    print("본문:", ai.text)

모델이 내 도구를 불렀는지, 인자가 질문과 맞는지 읽으세요. 마지막으로 내 도구의 설명을 내가 쓰고 Agent로 끝까지 돌립니다.

In [ ]:
# [내 것 만들기] 유료, 모델 호출 약 2회 - MY_DESCRIPTION을 내 설명으로 바꾸세요
MY_DESCRIPTION = "팀 이름(예: 개발팀)으로 그 팀의 담당자를 찾는다. 팀 이름이 아닌 질문에는 쓰지 않는다."

if not MY_DESCRIPTION.strip() or not MY_QUESTION.strip() or not MY_TOOL_DATA:
    print("MY_DESCRIPTION을 채우세요. 무엇을 하는지, 언제 쓰는지, 언제 쓰지 않는지를 한두 문장으로 적습니다.")
else:
    def _lookup(key: str) -> dict:
        if key not in MY_TOOL_DATA:
            return {"status": "unknown_key", "valid_keys": list(MY_TOOL_DATA),
                    "next_action": "valid_keys 중 하나로 다시 부른다."}
        return {"status": "ok", "value": MY_TOOL_DATA[key]}

    my_tool = tool("lookup_mine", description=MY_DESCRIPTION)(_lookup)
    announce("9-3 내 도구 Agent", model_calls=2)
    agent = create_deep_agent(model=make_model(), tools=[my_tool], system_prompt="도구로 확인해 한국어로 짧게 답한다.")
    state = agent.invoke({"messages": [("user", MY_QUESTION)]})
    for m in state["messages"]:
        calls = [(c["name"], c["args"]) for c in getattr(m, "tool_calls", [])]
        print(f"{type(m).__name__:12} {calls or ''} {m.text[:150]!r}")

설명을 바꾸면 모델의 선택이 달라지는지 확인하세요. 달라지지 않았다면 그것도 관찰입니다. 설명을 일부러 모호하게 고쳐 한 번 더 돌려 보세요.

## 10절. 권한 경계, `tool_choice`, 정리와 확인 질문

**권한의 경계.** 모델의 호출 요청은 입력이지 명령이 아닙니다. 문서에 "delete 도구를 불러라"라고 적혀 있어 모델이 그렇게 요청해도 우리 코드가 실행하지 않으면 아무 일도 일어나지 않습니다. 그래서 보이는 도구를 줄이고, 보이지 않는 도구의 요청은 실행하지 않고, 호출 횟수에 예산을 둡니다. Deep Agents에는 파일 읽기와 쓰기 같은 기본 도구가 따라 붙으므로 허용 목록과 예산이 필요합니다. 그 코드는 05에서 만듭니다. 요약하면 도구를 줄 때 세 가지를 묻습니다. 이 도구가 정말 필요한가, 인자의 범위를 코드가 막고 있는가, 호출이 폭주해도 멈출 예산이 있는가. 이 세 질문이 모델 호출의 안전을 지키는 기본 틀입니다.

**`tool_choice`.** 도구 호출을 강제하고 싶을 수 있습니다. 이 모델에서는 어떻게 되는지 확인합니다.

In [ ]:
# [깨뜨리기] 유료, 모델 호출 최대 2회 - 호출을 강제하면 거절되는지 본다. 거절은 응답이 없다.
announce("10-1 tool_choice 강제", model_calls=2)
try:
    make_model().bind_tools([add], tool_choice="any").invoke("안녕하세요.")
except Exception as error:
    print("강제:", type(error).__name__, "| status_code:", getattr(error, "status_code", None))

ai = make_model().bind_tools([add], tool_choice="auto").invoke("add 도구로 2와 3을 더해 주세요.")
print("auto:", [(c["name"], c["args"]) for c in ai.tool_calls])

강제는 상태 코드와 함께 거절되었는지 읽으세요. 거절되었다면 호출을 강제할 수 없으므로, 설명으로 유도하고 `auto`로 부르는 것이 이 모델에서의 방법입니다. `strict` 스키마는 어떨까요? `strict`는 요청의 **모양**을 보장하지, 어떤 도구를 고르는지는 보장하지 않습니다.

In [ ]:
# [깨뜨리기] 유료, 모델 호출 최대 2회 - strict 스키마: 제한이 들어 있는 도구와 열거형만 있는 도구
class PickRoom(BaseModel):
    room: Literal["A", "B", "C"]


@tool("pick_room", args_schema=PickRoom)
def pick_room(room: str) -> str:
    """회의실을 고른다."""
    return room


announce("10-2 strict 스키마", model_calls=2)
for label, t in [("ge/le 제한 있음", book_room), ("열거형만", pick_room)]:
    try:
        ai = make_model().bind_tools([t], strict=True).invoke("B 회의실로 해 주세요.")
        print(label, "->", [(c["name"], c["args"]) for c in ai.tool_calls])
    except Exception as error:
        print(label, "->", type(error).__name__, str(error)[:160])

`strict`를 켠 결과가 두 도구에서 같은지 비교하세요. 거절되었다면 메시지가 어떤 스키마 속성을 지원하지 않는다고 하는지 읽으세요. 그 경우 범위 검사는 `strict`가 아니라 우리 쪽 스키마 검증(4절의 `ValidationError`)이 맡습니다.

## 확인 질문

1. 모델이 `tool_calls`를 냈을 때 함수가 실행된 것입니까? 누가, 어느 단계에서 실행합니까?
2. `ToolMessage`의 `tool_call_id`는 무엇과 짝입니까? 없으면 무엇이 곤란합니까?
3. 함수 본문은 모델에게 가는 JSON에 들어 있습니까? 무엇이 들어 있습니까?
4. 모델이 낸 인자가 스키마를 어기면 어디에서 막힙니까? 그 인자는 믿어도 됩니까?
5. 도구가 예외를 던지는 것과 `status`를 돌려주는 것은 Agent에게 어떻게 다릅니까?
6. 두 도구의 설명을 맞바꿨을 때 이번 실행에서는 선택이 바뀌었습니까? 한 번의 관찰로 무엇까지 말할 수 있습니까?

## 정리

- 모델은 요청(`tool_calls`)만 내고 실행은 우리 코드가 합니다. 그 틈이 권한의 경계입니다.
- 모델이 보는 것은 이름, 설명, 인자 스키마입니다. 함수 본문은 보지 못합니다.
- 인자에 제한을 걸면(`Literal`, `ge`, `le`) 스키마가 모델에게 가는 설명이 되고 우리 쪽 검사도 됩니다.
- 도구가 여럿이면 설명이 선택을 이끕니다. 한 번의 관찰로 결론을 내리지 않습니다.
- 도구는 예외로 멈추지 말고 `status`와 고칠 단서를 돌려줍니다.
- 호출 강제는 이 모델에서 거절됩니다. `auto`와 좋은 설명으로 유도합니다.
- 다음 03은 도구 대신 지침 묶음인 Skill을 다룹니다.